In [4]:
# ==============================
# TRAINING (DEMO VERSION)
# ==============================

import numpy as np
import tensorflow as tf
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Clean dataset (~50 pairs for fast demo)
pairs = [
    ("hi", "hello"),
    ("hello", "hi there"),
    ("how are you", "i am fine"),
    ("what is your name", "i am a chatbot"),
    ("who are you", "i am a simple chatbot"),
    ("bye", "goodbye"),
    ("thanks", "you are welcome"),
    ("what is ai", "ai means artificial intelligence"),
    ("i am happy", "that is great"),
    ("i am sad", "i hope you feel better"),
]

inputs = [p[0] for p in pairs]
responses = [p[1] for p in pairs]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(inputs + responses)

vocab_size = len(tokenizer.word_index) + 1

X_seq = tokenizer.texts_to_sequences(inputs)
y_seq = tokenizer.texts_to_sequences(responses)

# Padding
max_len = max(max(len(seq) for seq in X_seq),
              max(len(seq) for seq in y_seq))

X = pad_sequences(X_seq, maxlen=max_len, padding='post')
y = pad_sequences(y_seq, maxlen=max_len, padding='post')

# One-hot encoding
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

# Model
model = Sequential([
    Embedding(vocab_size, 32, input_length=max_len),
    SimpleRNN(32, return_sequences=True),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')

# Train
model.fit(X, y, epochs=200, verbose=0)


In [5]:

# Save artifacts
model.save("model.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save max_len (IMPORTANT)
with open("config.pkl", "wb") as f:
    pickle.dump({"max_len": max_len}, f)

print("✅ Model, tokenizer, config saved!")

✅ Model, tokenizer, config saved!
